# Business ER — Phase 0: load + audit the facts on the FULL training data
Nothing here decides the pipeline. It only measures the facts the design rests on,
so every later choice is backed by a number you produced yourself.
Run top to bottom. Colab: High-RAM runtime recommended (>=25 GB); works on 12.7 GB with FRAC_TOK < 1.

In [ ]:
# CELL 0 — setup
!pip -q install rapidfuzz lightgbm anyascii regex pyarrow
import os, gc, time, re
import numpy as np, pandas as pd
import regex as rx

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/amazon_ml/student_resource/dataset'   # <-- edit to your path
    WORK = '/content/work'
else:
    DATA = os.environ.get('DATA', 'fake'); WORK = os.environ.get('WORK', 'work')
os.makedirs(WORK, exist_ok=True)
SEED = 0
FRAC_TOK = float(os.environ.get('FRAC_TOK', 0.3))   # token audit only: fraction of S1 CLUSTERS kept WHOLE (unbiased for per-cluster rates). Numbers audit uses 100%. Set 1.0 on High-RAM.
t0 = time.time()

In [ ]:
# CELL 1 — load TSV -> parquet, with an integrity check against raw line counts
def count_lines(p):
    n = 0
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): n += b.count(b'\n')
    return n

def load(split, name):
    p = f'{DATA}/{split}/{split}_{name}.tsv'
    df = pd.read_csv(p, sep='\t', dtype=str, keep_default_na=False, quoting=3, on_bad_lines='error')
    raw = count_lines(p) - 1
    assert len(df) == raw, f'{p}: parsed {len(df)} rows but file has {raw} data lines'
    return df

FILES = {}
for split in ['train', 'test']:
    for name in ['source1', 'source2', 'source3'] + (['ground_truth'] if split == 'train' else []):
        p = f'{DATA}/{split}/{split}_{name}.tsv'
        if not os.path.exists(p): print('missing (skipped):', p); continue
        df = load(split, name)
        out = f'{WORK}/{split}_{name}.parquet'; df.to_parquet(out, index=False); FILES[(split, name)] = out
        print(f'{split:5s} {name:12s} rows={len(df):>9,d} cols={list(df.columns)}')
        del df; gc.collect()

In [ ]:
# CELL 2 — normalisation used ONLY for the audit (the pipeline's normaliser is designed separately)
# Keeps Unicode letters, combining marks (Indic matras/viramas) and digits. Never NFKD-strips.
_keep = rx.compile(r'[^\p{L}\p{M}\p{N}\s]+')
_ws = rx.compile(r'\s+')
_nonlatin = rx.compile(r'(?!\p{Latin})\p{L}')
NULLS = {'', 'null', '<null>', 'n/a', 'na', 'none', 'nan'}

def norm(s):
    s = _keep.sub(' ', s.lower().replace('&', ' and '))
    return _ws.sub(' ', s).strip()

def rd(split, name):
    return pd.read_parquet(FILES[(split, name)])

s1 = rd('train', 'source1')
q = pd.concat([rd('train', 'source2').assign(src='S2'), rd('train', 'source3').assign(src='S3')], ignore_index=True)
gt = rd('train', 'ground_truth')
g = gt.assign(rid=gt['matched_entity_ids'].str.split(',')).explode('rid')
g = g.loc[g['rid'].notna() & (g['rid'] != ''), ['source1_entity_id', 'rid']].rename(columns={'source1_entity_id': 's1'})
PREFIX_BAD = int((~g['rid'].str[:3].isin(['S2-', 'S3-'])).sum())
g['src'] = g['rid'].str[:2]

# int64 ids: S1 -> number; S2 -> number; S3 -> number + 1e10  (memory: strings would need ~5 GB in the explodes below)
OFF = 10**10
def qid(ent):  # works on 'S2-123' / 'S3-123' strings
    n = ent.str[3:].astype('int64'); assert n.max() < OFF
    return n + np.where(ent.str[:2] == 'S3', OFF, 0)
s1['id'] = s1['entity_id'].str[3:].astype('int64'); assert s1['id'].is_unique
q['id'] = qid(q['entity_id']); assert q['id'].is_unique
g['s1'] = g['s1'].str[3:].astype('int64'); g['rid'] = qid(g['rid'])
gt_s1 = gt['source1_entity_id'].str[3:].astype('int64')
print(f'S1={len(s1):,}  S2+S3={len(q):,}  GT pairs={len(g):,}  GT rows={len(gt):,}')

In [ ]:
# CELL 3 — F1..F4: structural facts about the ground truth
R = {}
R['F1_rid_in_multiple_S1'] = int(g['rid'].duplicated().sum())                       # many-to-one?
R['F1_gt_ids_missing_from_sources'] = int((~g['rid'].isin(q['id'])).sum())
R['F1_S1_missing_from_gt'] = int((~s1['id'].isin(gt_s1)).sum())
R['F1_gt_rows_duplicated'] = int(gt_s1.duplicated().sum())
R['F1_gt_prefix_violations'] = PREFIX_BAD

cq = q.set_index('id')['country']; c1 = s1.set_index('id')['country']
R['F2_country_agreement_on_true_pairs'] = float((g['rid'].map(cq).values == g['s1'].map(c1).values).mean())

q['matched'] = q['id'].isin(g['rid'])
print(pd.Series(R).to_string())
print('\nF3 share of S2/S3 records that match NO S1 (decoy/unmatched rate):')
print((1 - q.groupby(['country', 'src'])['matched'].mean()).unstack().round(4))

k = g.groupby(['s1', 'src']).size().unstack(fill_value=0).reindex(s1['id'], fill_value=0)
k['n'] = k.sum(axis=1); k['country'] = c1
print('\nF4 matches per S1 (by country): singleton share, mean, max')
print(k.groupby('country')['n'].agg(singleton=lambda x: (x == 0).mean(), mean='mean', max='max').round(4))
print('F4 corr(n_S2, n_S3) per S1:', round(float(np.corrcoef(k['S2'], k['S3'])[0, 1]), 4))
print('F4 distribution of matches per S1:\n', k['n'].value_counts().sort_index().head(15).to_string())

In [ ]:
# CELL 4 — F5, F6: noise surface (empty fields, scripts) and S1 name ambiguity
def is_empty(s): return norm(s) in NULLS
for nm, df in [('S1', s1), ('S2/S3', q)]:
    e = df['business_address'].map(is_empty)
    print(f'F5 empty address {nm}:', e.groupby(df['country']).mean().round(4).to_dict())
for fld in ['business_name', 'business_address']:
    nl = q[fld].map(lambda s: bool(_nonlatin.search(s)))
    print(f'F5 non-Latin script in S2/S3 {fld}:', nl.groupby([q['country'], q['src']]).mean().round(4).to_dict())
    print(f'F5 non-Latin script in S1 {fld}:', s1[fld].map(lambda s: bool(_nonlatin.search(s))).groupby(s1['country']).mean().round(4).to_dict())

s1['nname'] = s1['business_name'].map(norm); s1['naddr'] = s1['business_address'].map(norm)
grp = s1.groupby(['country', 'nname'])['entity_id'].transform('size')
print('\nF6 share of S1 whose normalised name is shared with another S1:', (grp > 1).groupby(s1['country']).mean().round(4).to_dict())
print('F6 largest same-name group:', int(grp.max()),
      '| duplicate (name,address) pairs among S1:', int(s1.duplicated(['country', 'nname', 'naddr']).sum()))

In [ ]:
# CELL 5 — F7: is there a per-source ADDRESS VIEW? (the entity-centric test, full data)
# For each true pair (r, s): X = numbers of r not in S1 s.  A number x in X is "explained by a same-source sibling"
# if another positive of s from the same source contains x.  Baselines: other-source siblings; the same-source
# positives of a RANDOM other S1 of the same country.  If records were conditionally independent given S1,
# same-source ~ cross-source ~ random.
_num = re.compile(r'\d+')
def nums(s): return {int(x[-15:]) for x in _num.findall(s)}   # int64; zero-padding folded (0560 == 560)

addr_q = q.set_index('id')['business_address']
P = g.assign(n=g['rid'].map(addr_q).map(nums))[['s1', 'src', 'rid', 'n']].explode('n').dropna(subset=['n'])
P['n'] = P['n'].astype('int64')
S = s1[['id', 'business_address']].assign(n=lambda d: d['business_address'].map(nums)).explode('n').dropna(subset=['n'])
S = S[['id', 'n']].rename(columns={'id': 's1'}).drop_duplicates(); S['n'] = S['n'].astype('int64')
P = P.drop_duplicates(['rid', 'n'])
X = P.merge(S.assign(in_s1=1), on=['s1', 'n'], how='left'); X = X[X['in_s1'].isna()].drop(columns='in_s1')  # extra numbers

c_same = P.groupby(['s1', 'src', 'n']).size().rename('c_same')
c_all = P.groupby(['s1', 'n']).size().rename('c_all')
X = X.join(c_same, on=['s1', 'src', 'n']).join(c_all, on=['s1', 'n'])
X['exp_same'] = X['c_same'] >= 2                     # >=2 because r itself contributes 1
X['exp_cross'] = (X['c_all'] - X['c_same']) >= 1

rng = np.random.default_rng(SEED)
sib_same = g.groupby(['s1', 'src']).size().rename('m')          # positives per (S1, source)
n_pos = g.groupby('s1').size().rename('n_pos')                    # positives per S1
cnt = c1
perm = {}
for ctry, ids in g['s1'].drop_duplicates().groupby(g['s1'].drop_duplicates().map(cnt)):
    a = ids.values; perm.update(zip(a, rng.permutation(a)))
X['s1_rand'] = X['s1'].map(perm)
X = X.join(c_same.rename('c_rand'), on=['s1_rand', 'src', 'n'])
X['exp_rand'] = X['c_rand'].fillna(0) >= 1

X = X.join(sib_same, on=['s1', 'src']).join(n_pos, on='s1')
has_sib = X['m'] >= 2; has_cross = (X['n_pos'] - X['m']) >= 1
print(f'F7 extra (not-in-S1) address numbers: {len(X):,} occurrences over {X["rid"].nunique():,} positive records')
print(pd.DataFrame({'same_source_sibling': [X.loc[has_sib, 'exp_same'].mean()],
                    'cross_source_sibling': [X.loc[has_cross, 'exp_cross'].mean()],
                    'random_S1_same_source': [X.loc[has_sib, 'exp_rand'].mean()]}, index=['P(explained)']).round(4).T)
rec = X[has_sib].groupby('rid')['exp_same'].all()
print(f'F7 record level: positives with extra numbers AND a same-source sibling: {len(rec):,}; all extras explained: {rec.mean():.4f}')
print(f'F7 coverage: share of positives having >=1 same-source sibling: {(g.join(sib_same, on=["s1","src"])["m"] >= 2).mean():.4f}')
X['country'] = X['s1'].map(cnt)
print('F7 same-source explained rate by country:', X[has_sib].groupby('country')['exp_same'].mean().round(4).to_dict())
del P, S, X; gc.collect()

In [ ]:
# CELL 6 — F7b: same test for address tokens and name tokens (non-numeric, Latin-script only)
_lat = rx.compile(r'^[a-z]+$')
keep_s1 = set(g['s1'].drop_duplicates().sample(frac=FRAC_TOK, random_state=SEED)) if FRAC_TOK < 1 else None
gg = g if keep_s1 is None else g[g['s1'].isin(keep_s1)]
def toks(s): return {t for t in norm(s).split() if _lat.match(t) and len(t) > 1}
for fld in ['business_address', 'business_name']:
    fq = q.set_index('id')[fld]; f1 = s1.set_index('id')[fld]
    P = gg.assign(t=gg['rid'].map(fq).map(toks))[['s1', 'src', 'rid', 't']].explode('t').dropna(subset=['t'])
    P['t'] = P['t'].astype('category')
    S = pd.DataFrame({'s1': gg['s1'].drop_duplicates()}).assign(t=lambda d: d['s1'].map(f1).map(toks)).explode('t').dropna()
    S = S[S['t'].isin(P['t'].cat.categories)]; S['t'] = S['t'].astype(pd.CategoricalDtype(P['t'].cat.categories))
    X = P.merge(S.assign(in_s1=1), on=['s1', 't'], how='left'); X = X[X['in_s1'].isna()]
    cs = P.groupby(['s1', 'src', 't']).size().rename('c_same'); ca = P.groupby(['s1', 't']).size().rename('c_all')
    X = X.join(cs, on=['s1', 'src', 't']).join(ca, on=['s1', 't']).join(sib_same, on=['s1', 'src']).join(n_pos, on='s1')
    X['s1_rand'] = X['s1'].map(perm); X = X.join(cs.rename('c_rand'), on=['s1_rand', 'src', 't'])
    hs = X['m'] >= 2; hc = (X['n_pos'] - X['m']) >= 1
    print(f'F7b {fld}: extra tokens={len(X):,} | same-src sibling {(X.loc[hs,"c_same"]>=2).mean():.4f} | '
          f'cross-src sibling {((X.loc[hc,"c_all"]-X.loc[hc,"c_same"])>=1).mean():.4f} | random S1 {(X.loc[hs,"c_rand"].fillna(0)>=1).mean():.4f}')
    del P, S, X; gc.collect()
print(f'Phase 0 done in {time.time()-t0:.0f}s. Paste all printed output back.')